In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET_COL = 'Depression'
ID_COL = 'id'

train_path = '../data/train.csv'
test_path = '../data/test.csv'
submission_path = '../data/sample_submission.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())

Train shape: (140700, 20)
Test shape: (93800, 19)


,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


In [7]:
# 1. Tách feature và target
drop_cols = [ID_COL, 'Name']

X = train.drop(columns=[TARGET_COL] + drop_cols)
y = train[TARGET_COL]
X_test = test.drop(columns=drop_cols)
test_ids = test[ID_COL]

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
print(y.value_counts(normalize=True).sort_index())


X shape: (140700, 17)
X_test shape: (93800, 17)
Depression
0    0.818287
1    0.181713
Name: proportion, dtype: float64


In [8]:
#2. Chuẩn hoá categorical, xử lý missing theo vai trò và thêm missing indicator
ROLE_COL = 'Working Professional or Student'
STUDENT_ONLY_NUMERIC_COLS = ['Academic Pressure', 'CGPA', 'Study Satisfaction']
WORK_ONLY_NUMERIC_COLS = ['Work Pressure', 'Job Satisfaction']
WORK_ONLY_CATEGORICAL_COLS = ['Profession']
BINARY_YES_NO_COLS = [
    'Have you ever had suicidal thoughts ?',
    'Family History of Mental Illness',
]


def clean_string_columns(df):
    df = df.copy()
    obj_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        # sklearn SimpleImputer can fail on pandas pd.NA, so keep missing as np.nan.
        df[col] = df[col].astype('object')
        df[col] = df[col].where(df[col].notna(), np.nan)
        df[col] = df[col].str.strip()
        df[col] = df[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return df


def add_missing_indicators(df, cols):
    df = df.copy()
    for col in cols:
        df[f'{col}_is_missing'] = df[col].isna().astype(int)
    return df


def fill_structural_missing_by_role(df):
    df = df.copy()
    is_student = df[ROLE_COL].eq('Student')
    is_professional = df[ROLE_COL].eq('Working Professional')

    for col in STUDENT_ONLY_NUMERIC_COLS:
        if col in df.columns:
            df.loc[is_professional & df[col].isna(), col] = 0

    for col in WORK_ONLY_NUMERIC_COLS:
        if col in df.columns:
            df.loc[is_student & df[col].isna(), col] = 0

    for col in WORK_ONLY_CATEGORICAL_COLS:
        if col in df.columns:
            df.loc[is_student & df[col].isna(), col] = 'Not Applicable'

    return df


def map_yes_no_columns(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].map({'No': 0, 'Yes': 1})
    return df


def group_rare_categories(train_df, test_df, min_count=30):
    train_df = train_df.copy()
    test_df = test_df.copy()
    cat_cols = train_df.select_dtypes(include=['object', 'string']).columns.tolist()

    for col in cat_cols:
        counts = train_df[col].value_counts(dropna=True)
        keep_values = set(counts[counts >= min_count].index)

        train_mask = train_df[col].notna() & ~train_df[col].isin(keep_values)
        test_mask = test_df[col].notna() & ~test_df[col].isin(keep_values)
        train_df.loc[train_mask, col] = 'Other'
        test_df.loc[test_mask, col] = 'Other'

    return train_df, test_df


X_clean = clean_string_columns(X)
X_test_clean = clean_string_columns(X_test)

missing_cols = X_clean.columns[X_clean.isna().mean() > 0].tolist()
print('Columns with missing values before role-aware fill:', missing_cols)

X_clean = add_missing_indicators(X_clean, missing_cols)
X_test_clean = add_missing_indicators(X_test_clean, missing_cols)

X_clean = fill_structural_missing_by_role(X_clean)
X_test_clean = fill_structural_missing_by_role(X_test_clean)

X_clean = map_yes_no_columns(X_clean, BINARY_YES_NO_COLS)
X_test_clean = map_yes_no_columns(X_test_clean, BINARY_YES_NO_COLS)

X_clean, X_test_clean = group_rare_categories(X_clean, X_test_clean, min_count=30)

display(X_clean.head())

Columns with missing values before role-aware fill: ['Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Dietary Habits', 'Degree', 'Financial Stress']


,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,...,Family History of Mental Illness,Profession_is_missing,Academic Pressure_is_missing,Work Pressure_is_missing,CGPA_is_missing,Study Satisfaction_is_missing,Job Satisfaction_is_missing,Dietary Habits_is_missing,Degree_is_missing,Financial Stress_is_missing
0,Female,49.0,Ludhiana,Working Professional,Chef,0.0,5.0,0.00,0.0,2.0,...,0,0,1,0,1,1,0,0,0,0
1,Male,26.0,Varanasi,Working Professional,Teacher,0.0,4.0,0.00,0.0,3.0,...,0,0,1,0,1,1,0,0,0,0
2,Male,33.0,Visakhapatnam,Student,Not Applicable,5.0,0.0,8.97,2.0,0.0,...,0,1,0,1,0,0,1,0,0,0
3,Male,22.0,Mumbai,Working Professional,Teacher,0.0,5.0,0.00,0.0,1.0,...,1,0,1,0,1,1,0,0,0,0
4,Female,30.0,Kanpur,Working Professional,Business Analyst,0.0,1.0,0.00,0.0,1.0,...,1,0,1,0,1,1,0,0,0,0


In [9]:
remaining_missing = X_clean.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)

print('Remaining missing after role-aware fill:')
display(remaining_missing)

missing_indicator_cols = [c for c in X_clean.columns if c.endswith('_is_missing')]
display(X_clean[missing_indicator_cols].head())


Remaining missing after role-aware fill:


Profession            8763
Work Pressure           20
Job Satisfaction        17
Study Satisfaction      10
Academic Pressure        9
CGPA                     9
Dietary Habits           4
Financial Stress         4
Degree                   2
dtype: int64

,Profession_is_missing,Academic Pressure_is_missing,Work Pressure_is_missing,CGPA_is_missing,Study Satisfaction_is_missing,Job Satisfaction_is_missing,Dietary Habits_is_missing,Degree_is_missing,Financial Stress_is_missing
0,0,1,0,1,1,0,0,0,0
1,0,1,0,1,1,0,0,0,0
2,1,0,1,0,0,1,0,0,0
3,0,1,0,1,1,0,0,0,0
4,0,1,0,1,1,0,0,0,0


In [10]:
# 3.Chia train/val có stratify 
X_train, X_val, y_train, y_val = train_test_split(
    X_clean,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train target ratio:')
print(y_train.value_counts(normalize=True).sort_index())
print('\nValidation target ratio:')
print(y_val.value_counts(normalize=True).sort_index())


Train target ratio:
Depression
0    0.818284
1    0.181716
Name: proportion, dtype: float64

Validation target ratio:
Depression
0    0.818301
1    0.181699
Name: proportion, dtype: float64


In [11]:
# Tạo Preprocessing Pipeline cho tree-based models
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('onehot', make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols),
    ],
    remainder='drop',
)

preprocessor



Numeric columns: ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Profession_is_missing', 'Academic Pressure_is_missing', 'Work Pressure_is_missing', 'CGPA_is_missing', 'Study Satisfaction_is_missing', 'Job Satisfaction_is_missing', 'Dietary Habits_is_missing', 'Degree_is_missing', 'Financial Stress_is_missing']
Categorical columns: ['Gender', 'City', 'Working Professional or Student', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test_clean)

print('Processed train shape:', X_train_processed.shape)
print('Processed val shape:', X_val_processed.shape)
print('Processed test shape:', X_test_processed.shape)


Processed train shape: (112560, 131)
Processed val shape: (28140, 131)
Processed test shape: (93800, 131)


In [13]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)
X_val_processed_df = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index,
)

display(X_train_processed_df.head())


,num__Age,num__Academic Pressure,num__Work Pressure,num__CGPA,num__Study Satisfaction,num__Job Satisfaction,num__Have you ever had suicidal thoughts ?,num__Work/Study Hours,num__Financial Stress,num__Family History of Mental Illness,...,cat__Degree_MBA,cat__Degree_MBBS,cat__Degree_MCA,cat__Degree_MD,cat__Degree_ME,cat__Degree_MHM,cat__Degree_MSc,cat__Degree_Other,cat__Degree_PhD,cat__Degree_Unknown
3429,25.0,5.0,0.0,5.59,5.0,0.0,1.0,8.0,5.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
57741,20.0,3.0,0.0,8.27,4.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
83234,24.0,3.0,0.0,6.00,2.0,0.0,0.0,3.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136573,38.0,0.0,1.0,0.00,0.0,3.0,0.0,10.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
93261,24.0,4.0,0.0,8.04,4.0,0.0,1.0,10.0,3.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Chạy thử model RandomForest
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score

model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        (
            'classifier',
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
val_proba = model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, val_pred))
print('ROC-AUC:', roc_auc_score(y_val, val_proba))

              precision    recall  f1-score   support

           0       0.98      0.92      0.95     23027
           1       0.71      0.91      0.80      5113

    accuracy                           0.92     28140
   macro avg       0.85      0.91      0.87     28140
weighted avg       0.93      0.92      0.92     28140

ROC-AUC: 0.9708669533433447


In [15]:
# Tạo submission mẫu
final_model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        (
            'classifier',
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

final_model.fit(X_clean, y)
test_pred = final_model.predict(X_test_clean)

submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: test_pred})
display(submission.head())

submission.to_csv('submission_preprovc_random_forest.csv', index=False)

,id,Depression
0,140700,0
1,140701,0
2,140702,0
3,140703,1
4,140704,0
